## Introduction to Large Language Models (LLMs) & Prompt Engineering

Before we dive into prompt engineering, let's establish a foundational understanding of what Large Language Models (LLMs) are and how they work.

### What is an LLM?
Large Language Models are deep neural networks with billions of tunable parameters that have been trained on huge text corpora using "self supervision". Once this foundational training is complete, the LLM parameters are fine-tuned to complete specific tasks such as question answering, summarization, and machine translation.

At its core, an LLM is a system where "text goes in and text comes out". 
* **Tokens:** LLMs don't read words like we do; they convert words or partial words into numerical IDs called **tokens**.
* **Context Window:** The context window (or context length) is the maximum number of tokens an LLM will accept as input at one time.

### Demystifying LLM Text Generation

The Python cell below is a simple simulation, but it illustrates the core mechanic of how Large Language Models actually work: **autoregressive generation**. 

At their simplest level, LLMs operate on a basic principle: text goes in, and text comes out. However, they do not write entire sentences or paragraphs at once. Instead, they break text down into smaller chunks called **tokens** (which can be whole words, syllables, or even single characters).

Here is what our simulation is demonstrating about the LLM "brain":

1. **The Prompt:** This is your initial input. Everything the model generates is strictly based on this starting text. 
2. **The Prediction:** The model analyzes the prompt and calculates the most mathematically probable *single next token*.
3. **The Recursive Loop:** The model takes that single new token, attaches it to the end of your original prompt, and then feeds that *entire new string* back into itself to predict the next token. 

When you run the Python loop below, you are watching this recursive cycle in action. Every time the text updates, you are seeing the model's "context window" growing larger, token by token, until the task is complete.

In [1]:
import time
import sys
from IPython.display import clear_output

prompt = """Instructions: Write a poem that exactly rhymes with orange

Output:
"""

# The "tokens" the LLM generates one by one
tokens = ["As ", "I ", "sat ", "under ", "a ", "tree ", "looking ", "at ", "a ", "door hinge..."]

current_text = prompt

# Print the prompt first
print(current_text + "█")
time.sleep(1.5)

# Simulate the token-by-token generation
for token in tokens:
    current_text += token
    clear_output(wait=True)
    # The █ character simulates the blinking cursor!
    print(current_text + "█") 
    time.sleep(0.4) # Adjust this number to make it type faster or slower

clear_output(wait=True)
print(current_text)
print("\n[Generation Complete]")

Instructions: Write a poem that exactly rhymes with orange

Output:
As I sat under a tree looking at a door hinge...

[Generation Complete]


### The Transformer Architecture
Modern LLMs rely on the **Transformer** architecture, which fundamentally changed how AI processes language. Older models read text sequentially (word by word), making it hard to connect concepts at the beginning of a long paragraph to words at the end. Transformers solve this by reading the entire sequence at once and calculating the contextual meaning of words using a mechanism called **Self-Attention**.

#### A Quick Note on the Architecture Diagrams
If the diagrams below look like a daunting maze of math and boxes, don't worry! You do not need to memorize matrix multiplication, SoftMax functions, or the exact wiring of the neural network to use Large Language Models effectively. Here is how to read these diagrams conceptually:

**1. Self-Attention: The Context Map**
How does a model know what a pronoun refers to? Consider the sentence: *"The cat drank the milk because it was hungry."* We intuitively know "it" refers to the cat. But if we change just one word—*"The cat drank the milk because it was sweet"*—"it" suddenly refers to the milk.

Transformers solve this using "attention scores." In the diagram below, look at the lines connecting the words. When the model processes the word "it" in the first sentence, it calculates a high attention score (indicated by a darker color or thicker line) connecting "it" to "cat". In the second sentence, the math shifts, and "it" gets a high attention score connecting to "milk". This is how the model builds a numerical map of context!

![attention_score](./attention_score.png)<sup>[1]</sup>


**2. Multi-Head Attention: Multiple Perspectives**
Understanding a complex sentence requires looking at it from multiple angles simultaneously. One part of the model might pay attention to grammar (verbs connecting to nouns), while another part pays attention to emotional tone, and another to factual relationships. The architecture handles this using multiple attention "heads," running these calculations in parallel.
![attention_head](./attention_heads.png)<sup>[2]</sup>


**3. The Transformer Pipeline**
Modern generative LLMs utilize a **Decoder-only** model. When you zoom out, it is essentially just a tall stack of these Multi-Head Attention blocks. The input text is encoded into numbers, passed up through these parallel attention layers to deeply enrich the context, and then immediately used to predict the next logical word.
![transformer_arch](./transformer_arch.png)<sup>[2]</sup>


### Tying It All Together: Inside the Decoder
So, how do these three concepts connect in a modern LLM? Think of them as building blocks stacked inside one another:

**1. The Core Action: Self-Attention:** This is the fundamental math operation. The model looks at a specific word (like "it") and calculates its relationship scores with the other words in the prompt to figure out what it refers to.

**2. The Layer: Multi-Head Attention:** A single attention "head" might only be good at finding one type of relationship (like grammar). To get a full understanding, the model bundles several of these self-attention operations together into a "Multi-Head" layer so it can calculate grammar, factual context, and tone all at the exact same time.

**3. The Pipeline:** A single "Decoder Block" contains that Multi-Head Attention layer, followed by a standard feed-forward neural network. A modern LLM is simply dozens of these Decoder Blocks stacked on top of each other. 

**The Golden Rule of the Decoder: Masking:**
One critical rule in a Decoder-only model is **it cannot see the future.** Because its entire job is to predict the *next* word, when it calculates those self-attention scores, it applies a mathematical "mask" that hides any words that come after the current token. It can only build context using the words you provided in the prompt and the words it has already generated.

[1] “Transformers Explained Visually” Ketan Doshi Towards Data Science (2020)

[2] Elhage, N., et al. (2021). "A Mathematical Framework for Transformer Circuits." *Transformer Circuits Thread*.

## Prompt Engineering

A prompt is a set of instructions (including content like text, images, etc) provided to an LLM to perform a task.

Prompt Engineering is the process of crafting such the set of instructions that gets an LLM model to generate the desired outcome (i.e., solving the task).

Components of Well-Structured Prompts
- Role: The role the LLM should adopt.
- Task Description: The specific instruction or question.
- Context: Additional information needed for the task.
- Output Format: How the response should be structured.
- Examples (a.k.a. Few-Shot "Learning") (optional): Sample input/output pairs.

In [2]:
import os
%env HF_HOME={os.environ['SCRATCH']}/HFHOME
%env CUDA_HOME=/home1/apps/nvidia/Linux_aarch64/25.5/cuda/12.9/

env: HF_HOME=/scratch/07980/sli4/HFHOME
env: CUDA_HOME=/home1/apps/nvidia/Linux_aarch64/25.5/cuda/12.9/


### Prepare the model

To prepare the model for the tutorial, let's  download the weights from the [Hugging Face Hub](https://huggingface.co/). We will use the `AutoModelForCausalLM` class from 🤗 Transformers to load the model:

In [3]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, Mxfp4Config

quantization_config = Mxfp4Config(dequantize=True)
model_kwargs = dict(
    attn_implementation="eager",
    torch_dtype=torch.bfloat16,
    quantization_config=quantization_config,
    use_cache=False,
    device_map="auto",
)
model = AutoModelForCausalLM.from_pretrained("openai/gpt-oss-20b", **model_kwargs)
# Load the tokenizer
tokenizer = AutoTokenizer.from_pretrained("openai/gpt-oss-20b")

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

In [4]:
def generate_response(
    model, 
    tokenizer, 
    user_prompt: str,
    system_prompt: str = "You are a helpful assistant.",
    temperature: float = 0.5,
    max_new_tokens: int = 512
) -> str:
    """Sends a request to a Hugging Face model and returns a response."""
    
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt}
    ]

    input_ids = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        return_tensors="pt",
    ).to(model.device)

    do_sample = True

    output_ids = model.generate(
        input_ids, 
        max_new_tokens=max_new_tokens,
        temperature=temperature,
        do_sample=do_sample,
        pad_token_id=tokenizer.eos_token_id,
    )
   
    input_length = input_ids.shape[1]
    new_tokens = output_ids[:, input_length:]
    
    raw_response = tokenizer.batch_decode(new_tokens, skip_special_tokens=True)[0]
    if "assistantfinal" in raw_response:
        raw_response = raw_response.split("assistantfinal")[-1]
    elif "</think>" in raw_response:
        raw_response = raw_response.split("</think>")[-1]
        
    final_response = raw_response.replace(tokenizer.eos_token, "")
    return final_response.strip()

### 1. Vague vs. Precise Prompt

Let's see how an LLM's output changes when we start from a vague prompt, then adding a persona/role, and finally specifying the task.

In [5]:
# Vague prompt = vague result
vague_prompt = "Tell me about Albert Einstein."

vague_response = generate_response(
    model=model,
    tokenizer=tokenizer,
    user_prompt=vague_prompt, 
)
print(vague_response)

The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


**Albert Einstein (1879 – 1955)** was a German‑born theoretical physicist whose work fundamentally reshaped our understanding of space, time, and the universe. Here’s a concise overview of his life, achievements, and legacy.

| **Period** | **Key Events & Contributions** |
|------------|--------------------------------|
| **Early Life (1879‑1905)** | Born in Ulm, Germany, Einstein grew up in Munich. He was a curious, slightly rebellious student who disliked rote learning. After a brief stint at the Swiss Federal Polytechnic (ETH) in Zurich, he earned a teaching certificate in 1900. |
| **Early Career & “Annus Mirabilis” (1905)** | While working as a patent clerk in Bern, he published four groundbreaking papers in *Annalen der Physik*: 1) **Photoelectric Effect** – explained light as quanta, earning him the 1921 Nobel Prize. 2) **Brownian Motion** – provided evidence for atoms. 3) **Special Relativity** – introduced the constancy of the speed of light and the equivalence of mass and ene

In [6]:
# Add persona (role) to the vague prompt
vague_prompt_with_persona = f"""
You are a high-school physics teacher.
Tell me about Albert Einstein.
"""

response_with_persona = generate_response(
    model=model,
    tokenizer=tokenizer,
    user_prompt=vague_prompt_with_persona,
)
print(response_with_persona)

Hey there! Let’s dive into the life and legacy of one of the most iconic scientists of all time—**Albert Einstein**. Think of him as the “Einstein of the 20th century,” the guy who changed the way we think about space, time, and even the very fabric of reality. Below is a quick, high‑school‑friendly rundown that covers the essentials:

| Topic | Key Points |
|-------|------------|
| **Early Life** | • Born **14 March 1879** in Ulm, Germany.<br>• Family moved to Munich when he was 3, then to Aarau, Switzerland (where he attended school).<br>• Showed early curiosity—couldn’t resist asking “why?” about everything. |
| **Education** | • Studied at the **Swiss Federal Polytechnic** (now ETH Zürich) 1896‑1900.<br>• Graduated with a diploma in **physics and mathematics** (not a Ph.D., but he still got one later). |
| **Early Career** | • Worked as a **patent clerk** in Bern (1902‑1909). <br>• This job gave him time to think, which led to some of his most revolutionary ideas. |
| **Annus Mirab

In [7]:
precise_prompt = f"""
You are a high-school physics teacher.
Explain the role Albert Einstein played in the development of physics in 20th century.
Write two paragraphs. 
"""

precise_response = generate_response(
    model=model,
    tokenizer=tokenizer,
    user_prompt=precise_prompt
)
print(precise_response)

Albert Einstein’s most celebrated contribution to 20th‑century physics is the theory of relativity, which reshaped our understanding of space, time, and gravity. In 1905 he published the special relativity paper that introduced the famous equation \(E=mc^2\), showing that mass and energy are interchangeable and that the speed of light is an absolute limit. A decade later, his general theory of relativity described gravity not as a force but as the curvature of spacetime caused by mass and energy. These ideas resolved long‑standing puzzles, such as the anomalous precession of Mercury’s orbit, and opened a new framework for studying the cosmos, from black holes to the expanding universe.

Beyond relativity, Einstein’s work laid the groundwork for quantum mechanics and modern physics. His 1905 photoelectric effect paper provided the first solid evidence that light consists of discrete quanta—photons—thereby helping to establish quantum theory. He also introduced the concept of wave‑partic

In the case of the vague prompt, we can see that the output text covers various aspects of Albert Einstein's live without providing depth into any of those. Advancing forward, when we provide a role ("a high-school physics teacher"), the output changes its format and is "tailored" to the role/persona specified. Finally, with the precise prompt, we guide an LLM to produce a specific but more detailed response. While for some cases (e.g., famous personas) we might want to learn both perspectives (in breadth and depth), for AI applications we typically want to narrow down the scope of the task to get more specific outputs.

So, **Best Practice #1: Be specific and direct**.

### 2. Add Context

Any LLM model "knows" only what it was trained on. However, you can incorporate new information by providing it as a context.

In [8]:
# Source: https://tacc.utexas.edu/about/staff-directory/niall-gaffney/
context = """
NIALL GAFFNEY

Data and AI Directorate

Phone: 512-475-9504 | Email: ngaffney@tacc.utexas.edu

Education: B.A., M.A., Ph.D., Astronomy University of Texas at Austin

Niall Gaffney's background primarily revolves around the management and utilization of 
large inhomogeneous scientific datasets. Niall, who earned his B.A., M.A., and Ph.D. 
degrees in astronomy from The University of Texas at Austin, joined TACC in May 2013. 
Most of his focus has been on creating environments to foster better data practices 
from improving metadata, data processing, analysis, and reuse. He focuses on improving 
researchers' data practices to accelerate outcomes and better feed the Machine Learning 
and Artificial Intelligence applications which are becoming more broadly adopted in 
science and engineering research fields. Much of this stems from his 13 years as designer 
and developer for the archives at the Space Telescope Science Institute (STScI), which 
holds the data from the Hubble Space Telescope, Kepler, and James Webb Space Telescope 
missions.

He was also a leader in developing the Hubble Legacy Archive. This project harvested 
the 20+ years of Hubble Space Telescope data to create some of the most sensitive 
astronomical data products available for open research. Before his work at STScI, 
Niall was worked as "the friend of the telescope" for the Hobby Eberly Telescope (HET) 
project at the McDonald Observatory in west Texas. This was the start of his work in 
planning experiments and then cataloging the data the HET produced.
"""

In [9]:
# First let's see what a model "knows" about Niall Gaffney.
# In other words, do NOT provide any context first. 
no_context_prompt = "Who is Niall Gaffney?"

response_wo_context = generate_response(
    model=model,
    tokenizer=tokenizer,
    user_prompt=no_context_prompt,
)

print(response_wo_context)

analysisThe user asks: "Who is Niall Gaffney?" We need to answer. Likely a person. Could be a public figure. Let's search memory: Niall Gaffney is a former Irish Gaelic footballer? No, maybe Niall Gaffney is a musician? Let's think. There's a Niall Gaffney who is a former Gaelic footballer for Dublin? Actually, Niall Gaffney is a Gaelic footballer, played for Dublin. He played for St. Vincent's club. He also played for Dublin county team. He was a center back. He was part of the 2009 All-Ireland. Also he might be a former Gaelic footballer turned coach. There's also a Niall Gaffney who is a former Irish rugby union player? Not sure. There's a Niall Gaffney who is a former Gaelic footballer, and also a former Gaelic footballer turned coach. He may have been a former Dublin senior footballer, played in the 2009 All-Ireland.

Alternatively, Niall Gaffney could be a musician or a journalist. Let's check: There is a Niall Gaffney, a former Irish rugby union player? I recall a Niall Gaffney 

In [10]:
# Now, let's add the context.
prompt_with_context = no_context_prompt + f"\nContext: {context}"

response_with_context = generate_response(
    model=model,
    tokenizer=tokenizer,
    user_prompt=prompt_with_context,
)

print(response_with_context)

**Niall Gaffney** is a senior data‑science professional and the head of the Data and AI Directorate at the Texas Advanced Computing Center (TACC) at the University of Texas at Austin.  

- **Education** – B.A., M.A., and Ph.D. in Astronomy from the University of Texas at Austin.  
- **Career at TACC** – Joined in May 2013; leads initiatives to build data‑centric research environments, improve metadata, data processing, and reuse, and to accelerate scientific discovery through machine‑learning and AI tools.  
- **Previous experience** – 13 years at the Space Telescope Science Institute (STScI), where he designed and developed archives for Hubble, Kepler, and James Webb data.  
- **Key projects** – Played a pivotal role in creating the Hubble Legacy Archive, which harvested more than 20 years of Hubble data into highly sensitive, openly available products.  
- **Earlier work** – Served as “the friend of the telescope” for the Hobby‑Eberly Telescope (HET) project at McDonald Observatory, 

We've seen that by providing context, we can "add" new information/knowledge to a model without retraining it. What's more, by providing context we can also reduce the occurrence of hallucinations [1]. However, note that adding the context does NOT guarantee that a model will strictly follow it [1].

Also, adding additional instructions like "generate response based on the context provided" to the prompt can also be helpful.

**Best Practice #2: Add specific context to your prompts when applicable.**

[1] Huyen, C. (2024). Prompt Engineering. In AI Engineering: Building Applications with Foundation Models. (pp. 211-252) O'Reilly Media, Inc.

### 3. Prompt Chaining

Break complex tasks into simpler subtasks. Use each LLM's output as a **context** for the next prompt/step.

In [11]:
topic = "the impact of Albert Einstein on philosophy of 20th century"

# Step 1: Create an outline for the blog post
prompt_step_1 = f"Come up with a 3 point outline for a post about: {topic}"

response_step_1 = generate_response(
    model=model,
    tokenizer=tokenizer,
    user_prompt=prompt_step_1,
)

print(response_step_1)

**Outline: The Impact of Albert Einstein on 20th‑Century Philosophy**

1. **Re‑shaping Ontology and Metaphysics**  
   - *Relativistic space‑time* overturns the absolute notions of time and space that underpinned classical metaphysics.  
   - Einstein’s theory forces philosophers to reconsider the nature of causality, determinism, and the possibility of an objective reality independent of observers.

2. **Transforming the Philosophy of Science**  
   - *Theory‑laden observation* and the empirical‑theoretical interplay highlighted in Einstein’s work fuel debates on realism vs. instrumentalism.  
   - His methodological insights (e.g., the role of symmetry, simplicity, and unification) influence logical empiricism, the Copenhagen interpretation, and later developments in the philosophy of physics.

3. **Influencing Ethical, Political, and Humanistic Thought**  
   - Einstein’s public stance on pacifism, civil rights, and the responsibility of scientists reshapes discussions on the social

In [12]:
# Step 2: Write introduction using the outline
prompt_step_2=f"""
Using the following OUTLINE, write an introduction paragraph with 80-100 words.
OUTLINE: {response_step_1}
Hook the reader with a surprising fact in the first sentence.
"""

response_step_2 = generate_response(
    model=model,
    tokenizer=tokenizer,
    user_prompt=prompt_step_2,
)

print(response_step_2)

analysisWe need to write an introduction paragraph 80-100 words, with a surprising fact in first sentence. Outline: Impact of Einstein on 20th-century philosophy. Must hook reader with surprising fact. 80-100 words. Let's craft about 90 words. Use surprising fact: "Did you know that Einstein once said he was 'more of a philosopher than a physicist'?" Or "Einstein's equations predicted a phenomenon that would later be named after him: the 'Einstein ring'?" Or "Surprisingly, Einstein’s work spurred debates that shaped not just physics, but the very foundations of metaphysics." Let's choose: "Surprisingly, Einstein’s equations predicted a phenomenon that would later be named after him: the 'Einstein ring'." That hooks. Then mention impact on ontology, philosophy of science, ethics. 90 words. Let's count. We'll write 90 words. Count words. Let's draft:

"Surprisingly, Einstein’s equations predicted a phenomenon that would later be named after him: the ‘Einstein ring.’  Yet his influence ex

In [13]:
# Step 3: Come up with a few options for the title
prompt_step_3 = f"""
Based on the INTRODUCTION below, come up with 3 catchy blog post titles.
INTRODUCTION: {response_step_2}
Format your output as 3 bullet points.
""" 
response_step_3 = generate_response(
    model=model,
    tokenizer=tokenizer,
    user_prompt=prompt_step_3,
)

print(response_step_3)

- **“From Einstein Rings to Reality: How Physics Rewrote 20th‑Century Philosophy”**  
- **“Beyond the Cosmos: Einstein’s Theory of Relativity and the Philosophical Revolution”**  
- **“The Einstein Effect: How a Physicist Became the Philosopher of Modern Thought”**


Main motivation behind the prompt chaining technique: a few smaller prompts are better than one that is a giant one. If your application require solving complex tasks with multiple steps, divide them into smaller subtasks by introducing smaller prompts and chain LLM's outputs together with the following prompts.

Remember: When working with LLMs, simpler instructions are better than complex ones.

**Best Practice #3: Break complex prompts into smaller ones.**